In [ ]:
-- Customer Analysis using COMMON TABLE EXPRESSION
-- Non-Recursive CTE 
-- Q. Step1 :  find the total sales prr customers
-- Step2 : find the last order date for each customer 
-- Step3 : Rank customers based on total sales per customer
WITH cte_totalsales AS
(
    SELECT
        CustomerID,
        SUM(Sales) AS total_sales
    from Sales.Orders
    GROUP BY CustomerID
)
-- Step2 : find the last order date for each customer 
,
cte_lastorder AS
(
    SELECT
        CustomerID,
        MAX(OrderDate) as last_order
    FROM Sales.Orders
    GROUP BY CustomerID
)
-- Step3 : Rank customers based on total sales per customer
,
cte_customerRank as 
(
    SELECT
    customerid,
    total_sales,
    RANK() OVER(order by total_sales DESC) AS customer_rank
    FROM cte_totalsales
)
-- Segment customer based on their total sales 
, cte_customersegment  AS
(
    SELECT
        CustomerID,
        total_sales,
        CASE WHEN total_sales > 100 THEN 'High'
            WHEN total_sales > 80 THEN 'Medium'
            ELSE 'Low'
        END Customersegment 
    FROM cte_totalsales
)
-- Main Query
SELECT
    c.CustomerID,
    c.FirstName,
    c.LastName,
    cts.total_sales,
    clo.last_order,
    cr.customer_rank,
    cs.Customersegment 
from Sales.Customers AS c 
LEFT JOIN cte_totalsales as cts  
on cts.CustomerID = c.CustomerID
LEFT JOIN cte_lastorder as clo 
on clo.CustomerID = c.CustomerID
LEFT JOIN cte_customerRank as cr  
on cr.CustomerID = c.CustomerID
LEFT JOIN cte_customersegment cs  
on cs.CustomerID = c.CustomerID


(5 rows affected)

CustomerID | FirstName | LastName | total_sales | last_order | customer_rank | Customersegment
-----------+-----------+----------+-------------+------------+---------------+----------------
1          | Jossef    | Goldberg | 110         | 2025-02-15 | 2             | High           
2          | Kevin     | Brown    | 55          | 2025-03-10 | 4             | Low            
3          | Mary      | NULL     | 125         | 2025-03-15 | 1             | High           
4          | Mark      | Schwarz  | 90          | 2025-02-18 | 3             | Medium         
5          | Anna      | Adams    | NULL        | NULL       | NULL          | NULL           
(5 rows)

Total execution time: 00:00:00.023

In [ ]:
-- Recursive CTE
-- Generate a sequence of a number from 1 to 20
WITH series AS(
    -- ANCHOR QUERY
    SELECT
    1 AS Mynumber
    UNION ALL

    -- RECURSIVE QUERY
    SELECT
    Mynumber + 1
    FROM series
    WHERE Mynumber < 20
)
-- MAIN QUERY
SELECT *
from series
OPTION (MAXRECURSION 5000) -- WE CAN CONTROL THE NUMBER OF ITERATION, THE DEFAULT IS 100

(20 rows affected)

Mynumber
--------
1       
2       
3       
4       
5       
6       
7       
8       
9       
10      
11      
12      
13      
14      
15      
16      
17      
18      
19      
20      
(20 rows)

Total execution time: 00:00:00.016

In [ ]:
-- RECURSIVE QUERY EXAMPLE -- 2
-- Q. SHOW THE EMPLOYEE HEIRARCHY BY DISPLAYING EACH EMPLOYEE'S LEVEL WITHIN THE ORGINIZATION. 

WITH CTE_EMP_HIERARCHY AS
(
    -- ANCHOR QUERY
    SELECT 
        EmployeeID,
        FirstName,
        ManagerID,
        1 AS Level 
    FROM Sales.Employees
    WHERE ManagerID IS NULL 
    UNION ALL
    -- RECURSIVE QUERY
    SELECT
        e.EmployeeID,
        e.FirstName,
        e.ManagerID,
        Level + 1
    FROM Sales.Employees AS e
    INNER JOIN CTE_EMP_HIERARCHY CEH  
    ON e.ManagerID = CEH.EmployeeID
)

-- MAIN QUERY

SELECT
*
FROM CTE_EMP_HIERARCHY


(5 rows affected)

EmployeeID | FirstName | ManagerID | Level
-----------+-----------+-----------+------
1          | Frank     | NULL      | 1    
2          | Kevin     | 1         | 2    
3          | Mary      | 1         | 2    
5          | Carol     | 3         | 3    
4          | Michael   | 2         | 3    
(5 rows)

Total execution time: 00:00:00.010